# AgentCache Mac Ablation Study

This notebook orchestrates the Mac/vLLM-Metal AgentCache ablation study. It is intentionally a report and control surface: the actual benchmark work runs in fresh subprocesses through scripts in `mac_ablation_study/scripts/`.

Target hardware: Apple M3 Max with 96 GB unified memory.

The first study uses the Python coding-agent system prompt domain. LMCache is not part of the Mac setup; native vLLM-Metal prefix caching is the warm-cache baseline.

In [ ]:
from __future__ import annotations

import json
import os
import subprocess
from pathlib import Path

try:
    import pandas as pd
except ImportError:
    pd = None

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None


def find_metal_root() -> Path:
    start = Path.cwd().resolve()
    for cand in (start, *start.parents):
        if (cand / 'vllm_metal').is_dir() and (cand / 'pyproject.toml').is_file():
            return cand
    raise FileNotFoundError('Run this notebook from inside the vllm-metal repo.')


def find_agentcache_root(metal_root: Path) -> Path:
    env_root = os.environ.get('AGENTCACHE_ROOT')
    if env_root:
        root = Path(env_root).expanduser().resolve()
        if (root / 'agentcache_compression').is_dir():
            return root
    for cand in (metal_root, *metal_root.parents):
        if (cand / 'agentcache_compression').is_dir():
            return cand
    raise FileNotFoundError('Could not find agentcache_compression/.')


METAL_ROOT = find_metal_root()
STUDY_ROOT = METAL_ROOT / 'mac_ablation_study'
SCRIPTS = STUDY_ROOT / 'scripts'
RESULTS = STUDY_ROOT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)
VENV_PY = METAL_ROOT / '.venv-vllm-metal' / 'bin' / 'python'
PYTHON = VENV_PY if VENV_PY.exists() else Path(os.sys.executable)

AGENTCACHE_ROOT = find_agentcache_root(METAL_ROOT)
AC = AGENTCACHE_ROOT / 'agentcache_compression'
PROMPTS = AC / 'prompts'
EVAL = AC / 'data' / 'python_agent_eval.jsonl'
AC_CENTROIDS = AC / 'centroids'

LLAMA_MODEL = 'mlx-community/Llama-3.2-1B-Instruct-bf16'
LLAMA_N = 128
LLAMA_K = AC_CENTROIDS / 'N128_2000_K.npy'
LLAMA_V = AC_CENTROIDS / 'N128_2000_V.npy'

RUN_LIVE = False  # Set True to execute benchmark scripts.

print('METAL_ROOT =', METAL_ROOT)
print('STUDY_ROOT =', STUDY_ROOT)
print('PYTHON     =', PYTHON)
print('AC         =', AC)
print('EVAL exists:', EVAL.exists())
print('Llama centroid exists:', LLAMA_K.exists(), LLAMA_V.exists())

## Supported Baselines

| Condition | Mac / vLLM-Metal support | Notes |
|---|---:|---|
| `cold` | yes | Full system prompt is physically sent and fully prefills every run. |
| `prefix_cache` / native APC | yes | vLLM-Metal native prefix caching; this is the warm-cache baseline on Mac. |
| `centroid` | yes | AgentCache centroid KV injection through the vLLM-Metal paged KV cache. |
| `centroid + prefix_cache` | yes | Centroid handles the synthetic prefix; native prefix caching can help repeated/growing history. |
| `LMCache` | no | LMCache is not currently supported by the local Metal setup. |
| `LMCache + centroid` | no | CUDA/Linux combined baseline only, unless a Metal/MLX LMCache connector is ported. |

## System Prompt Inputs

The first study uses the Python coding-agent prompt family. The 2000-token prompt is the main trained target for the existing Llama N=128 centroid. Search-agent prompts exist in the repo, but they are a separate domain and need their own matching centroids before making quality claims.

In [ ]:
prompt_rows = []
for length in (200, 500, 1000, 2000):
    path = PROMPTS / f'{length}_python_agent_system.txt'
    text = path.read_text()
    prompt_rows.append({
        'length_label': length,
        'file': path.name,
        'words': len(text.split()),
        'chars': len(text),
    })

if pd:
    display(pd.DataFrame(prompt_rows))
else:
    print(json.dumps(prompt_rows, indent=2))

## Runner Helpers

The notebook prints commands when `RUN_LIVE = False`. Set `RUN_LIVE = True` to run them. Each runner script starts fresh Python/vLLM processes and writes JSON/JSONL into `mac_ablation_study/results/`.

In [ ]:
def run_script(script_name: str, *args: object) -> None:
    cmd = [str(PYTHON), str(SCRIPTS / script_name), *[str(a) for a in args]]
    print('+ ' + ' '.join(cmd))
    if RUN_LIVE:
        env = os.environ.copy()
        venv_bin = VENV_PY.parent
        if venv_bin.exists():
            env['PATH'] = f'{venv_bin}{os.pathsep}{env.get("PATH", "")}'
        subprocess.run(cmd, cwd=METAL_ROOT, env=env, check=True)
    else:
        print('DRY RUN: set RUN_LIVE = True to execute')


def load_json(path: Path):
    if not path.exists():
        print(f'No result yet: {path}')
        return None
    return json.loads(path.read_text())


def show_rows(rows):
    if pd:
        display(pd.DataFrame(rows))
    else:
        print(json.dumps(rows, indent=2))

## Experiment 1: RoPE Parity Gate

Run this before trusting any quality output from a real centroid. It verifies that the injected K tensors are rotated with the model's own RoPE semantics.

In [ ]:
run_script(
    'run_rope_parity.py',
    '--model', LLAMA_MODEL,
    '--centroid-k', LLAMA_K,
    '--centroid-v', LLAMA_V,
    '--n', LLAMA_N,
)

## Experiment 2: Cold Full Prompt vs Centroid Injection

This is the main quality-capable Llama-1B run: cold full prompt against the real N=128 centroid trained for the Python-agent domain.

In [ ]:
run_script(
    'run_cold_inject.py',
    '--model', LLAMA_MODEL,
    '--tag', 'llama1b',
    '--n', LLAMA_N,
    '--centroid-k', LLAMA_K,
    '--centroid-v', LLAMA_V,
    '--prompt-lengths', '200,500,1000,2000',
    '--ttft-reps', 5,
)

cold = load_json(RESULTS / 'llama1b_cold.json')
inject = load_json(RESULTS / 'llama1b_inject_N128.json')

rows = []
if cold:
    for item in cold.get('ttft', []):
        rows.append({'mode': 'cold', **item})
if inject:
    for item in inject.get('ttft', []):
        rows.append({'mode': 'inject', **item})
show_rows(rows)

In [ ]:
if plt and rows:
    cold_rows = [r for r in rows if r['mode'] == 'cold']
    inj_rows = [r for r in rows if r['mode'] == 'inject']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([r['prompt_tokens'] for r in cold_rows], [r['ttft_ms'] for r in cold_rows], 'o-', label='cold full prompt')
    if inj_rows:
        ax.axhline(inj_rows[0]['ttft_ms'], linestyle='--', label='centroid inject')
    ax.set_xlabel('physical prompt tokens')
    ax.set_ylabel('TTFT proxy (ms)')
    ax.set_title('Cold full prompt vs centroid injection')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

## Experiment 3: Long-Context TTFT Scaling

This is TTFT-only. The cold prompt is synthetically extended from the Python-agent prompt. The injected line is a flat reference because the physical prompt remains approximately `N + user_tokens`.

In [ ]:
inject_ms = None
if inject:
    inject_ms = float(inject['ttft'][0]['ttft_ms'])
else:
    inject_ms = 23.2  # cached historical Llama-1B N=128 reference

run_script(
    'run_longctx.py',
    '--model', LLAMA_MODEL,
    '--tag', 'llama1b',
    '--inject-ms', inject_ms,
    '--lengths', '1000,2000,4000,8000,12000,16000',
)

longctx = load_json(RESULTS / 'llama1b_longctx.json')
long_rows = longctx.get('rows', []) if longctx else []
show_rows(long_rows)

In [ ]:
if plt and long_rows:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([r['ctx_tokens'] for r in long_rows], [r['cold_ms'] for r in long_rows], 'o-', label='cold full prompt')
    ax.plot([r['ctx_tokens'] for r in long_rows], [r['inject_ms'] for r in long_rows], '--', label='centroid inject')
    ax.set_xlabel('context tokens')
    ax.set_ylabel('TTFT proxy (ms)')
    ax.set_title('Long-context TTFT scaling')
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot([r['ctx_tokens'] for r in long_rows], [r['speedup'] for r in long_rows], 'o-')
    ax.axhline(1.0, color='gray', linestyle='--')
    ax.set_xlabel('context tokens')
    ax.set_ylabel('speedup vs injected')
    ax.set_title('Speedup grows with system prompt length')
    ax.grid(alpha=0.3)
    plt.show()

## Experiment 4: Model-Size Sweep

This is TTFT-only unless the selected model has a matching trained centroid. The script uses real centroids where available and generates dummy centroids for shape-correct timing on other models.

In [ ]:
run_script(
    'run_model_sweep.py',
    '--models', 'qwen05,llama1b,llama3b,qwen7b,qwen14b',
    '--lengths', '1000,2000,4000,8000,12000,16000',
)

sweep = load_json(RESULTS / 'model_sweep_summary.json') or []
show_rows(sweep)

In [ ]:
if plt and sweep:
    fig, ax = plt.subplots(figsize=(9, 5))
    for item in sweep:
        data = load_json(Path(item['longctx_result']))
        if not data:
            continue
        r = data['rows']
        ax.plot([x['ctx_tokens'] for x in r], [x['cold_ms'] for x in r], 'o-', label=f"{item['tag']} cold")
        ax.axhline(item['inject_ms'], linestyle='--', alpha=0.5, label=f"{item['tag']} inject")
    ax.set_xlabel('context tokens')
    ax.set_ylabel('TTFT proxy (ms)')
    ax.set_title('Cold-vs-inject TTFT by model size')
    ax.grid(alpha=0.3)
    ax.legend(ncol=2, fontsize=8)
    plt.show()

## Experiment 5: Native Prefix Cache Baseline

This optional multi-turn run compares `cold` against `warm_apc` using vLLM-Metal native prefix caching. Add `synthetic` to `--modes` when you want the centroid + native-prefix-cache condition as well.

In [ ]:
run_script(
    'run_prefix_cache_multiturn.py',
    '--model', LLAMA_MODEL,
    '--modes', 'cold,warm_apc',
    '--n-conversations', 2,
    '--turns-per-conv', 5,
    '--max-tokens', 256,
    '--overwrite',
)

mt_path = RESULTS / 'prefix_cache_multiturn.jsonl'
if mt_path.exists():
    mt_rows = [json.loads(line) for line in mt_path.read_text().splitlines() if line.strip()]
    show_rows(mt_rows[:10])
else:
    print(f'No multi-turn result yet: {mt_path}')

## Reporting Checklist

- Report TTFT medians, not single runs.
- Keep quality claims separate from TTFT-only dummy-centroid measurements.
- Label native prefix caching as the Mac warm-cache baseline.
- Do not claim LMCache support on Metal.
- For the first written result, focus on the Python coding-agent domain and the Llama-1B N=128 real centroid.